<a href="https://colab.research.google.com/github/inhajourney/AIFFEL_quest_cr/blob/main/VGG16_optimizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
  !pip install colab-ssh --upgrade

In [4]:
  import torch
  import torch.nn as nn
  import torch.optim as optim
  from torch.utils.data import DataLoader
  from torchvision import datasets, transforms, models
  import time
  import json

  BATCH_SIZE = 64
  EPOCHS = 10
  LR = 0.001
  DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  print(f"Using device: {DEVICE}")

  transform_train = transforms.Compose([
      transforms.Resize(224),
      transforms.RandomHorizontalFlip(),
      transforms.ToTensor(),
      transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
  ])
  transform_test = transforms.Compose([
      transforms.Resize(224),
      transforms.ToTensor(),
      transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
  ])

  print("Downloading STL-10 dataset...")
  train_dataset = datasets.STL10(root="./data", split="train", download=True, transform=transform_train)
  test_dataset = datasets.STL10(root="./data", split="test", download=True, transform=transform_test)
  train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
  test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
  print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")

  def create_vgg16():
      model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
      model.classifier[6] = nn.Linear(4096, 10)
      return model.to(DEVICE)

  def train_epoch(model, loader, criterion, optimizer):
      model.train()
      total_loss, correct, total = 0, 0, 0
      for inputs, targets in loader:
          inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
          optimizer.zero_grad()
          outputs = model(inputs)
          loss = criterion(outputs, targets)
          loss.backward()
          optimizer.step()
          total_loss += loss.item()
          _, pred = outputs.max(1)
          total += targets.size(0)
          correct += pred.eq(targets).sum().item()
      return total_loss / len(loader), 100. * correct / total

  def evaluate(model, loader, criterion):
      model.eval()
      total_loss, correct, total = 0, 0, 0
      with torch.no_grad():
          for inputs, targets in loader:
              inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
              outputs = model(inputs)
              loss = criterion(outputs, targets)
              total_loss += loss.item()
              _, pred = outputs.max(1)
              total += targets.size(0)
              correct += pred.eq(targets).sum().item()
      return total_loss / len(loader), 100. * correct / total

  def run_experiment(name, opt_class, **kwargs):
      print(f"\n{'='*50}\nTraining with {name}\n{'='*50}")
      model = create_vgg16()
      criterion = nn.CrossEntropyLoss()
      optimizer = opt_class(model.parameters(), lr=LR, **kwargs)
      history = []
      start = time.time()
      for epoch in range(EPOCHS):
          tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer)
          te_loss, te_acc = evaluate(model, test_loader, criterion)
          history.append((tr_loss, tr_acc, te_loss, te_acc))
          print(f"Epoch {epoch+1}/{EPOCHS} | Train: {tr_acc:.2f}% | Test: {te_acc:.2f}%")
      elapsed = time.time() - start
      print(f"Time: {elapsed:.2f}s, Final Test Acc: {te_acc:.2f}%")
      return {"history": history, "time": elapsed, "final_acc": te_acc}

  results = {}
  results["SGD"] = run_experiment("SGD", optim.SGD, momentum=0.9)
  results["Adam"] = run_experiment("Adam", optim.Adam)
  results["AdamW"] = run_experiment("AdamW", optim.AdamW)

  print("\n" + "="*60 + "\nFINAL RESULTS\n" + "="*60)
  for name, r in results.items():
      print(f"{name}: {r['final_acc']:.2f}% ({r['time']:.1f}s)")

Using device: cuda


100%|██████████| 2.64G/2.64G [00:42<00:00, 62.5MB/s]


Train: 5000, Test: 8000

Training with SGD
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 189MB/s]


Epoch 1/10 | Train: 82.42% | Test: 94.70%
Epoch 2/10 | Train: 95.64% | Test: 95.46%
Epoch 3/10 | Train: 97.54% | Test: 94.95%
Epoch 4/10 | Train: 97.84% | Test: 95.89%
Epoch 5/10 | Train: 98.68% | Test: 95.76%
Epoch 6/10 | Train: 99.02% | Test: 95.47%
Epoch 7/10 | Train: 99.30% | Test: 96.04%
Epoch 8/10 | Train: 99.44% | Test: 95.62%
Epoch 9/10 | Train: 99.54% | Test: 95.39%
Epoch 10/10 | Train: 99.42% | Test: 95.74%
Time: 1104.81s, Final Test Acc: 95.74%

Training with Adam
Epoch 1/10 | Train: 15.38% | Test: 20.38%
Epoch 2/10 | Train: 28.90% | Test: 32.35%
Epoch 3/10 | Train: 36.22% | Test: 38.15%
Epoch 4/10 | Train: 40.46% | Test: 42.36%
Epoch 5/10 | Train: 42.56% | Test: 41.92%
Epoch 6/10 | Train: 47.22% | Test: 44.36%
Epoch 7/10 | Train: 49.14% | Test: 47.79%
Epoch 8/10 | Train: 49.70% | Test: 49.48%
Epoch 9/10 | Train: 54.46% | Test: 47.88%
Epoch 10/10 | Train: 54.86% | Test: 49.40%
Time: 1075.04s, Final Test Acc: 49.40%

Training with AdamW
Epoch 1/10 | Train: 18.10% | Test: 30.0

# Learning Rate 수정
현상: Adam, AdamW 값이 거의 학습이 안 되는 수준. Random보다 조금 나은 정도

원인: Adam/AdamW는 내부적으로 learning rate를 자동 조절하는데, 초기 lr=0.001이 너무 크거나 잘못된 스케일일 가능성
pre-trained VGG16을 fine-tuning할 때 이미 학습된 가중치들이 있음
너무 큰 lr로 업데이트하면 → 기존 지식을 다 망가뜨림
Adam/AdamW가 lr=0.001에서 폭발적으로 업데이트해서 망가진 것 같음

시도: Learning Rate 0.0001로 조정

In [6]:
def run_experiment(name, opt_class, custom_lr=None, **kwargs):
    print(f"\n{'='*50}\nTraining with {name}\n{'='*50}")
    model = create_vgg16()
    criterion = nn.CrossEntropyLoss()

    lr = custom_lr if custom_lr else LR  # custom_lr 없으면 기본값
    optimizer = opt_class(model.parameters(), lr=lr, **kwargs)

    # ... 나머지 코드 동일

In [7]:
results["SGD"] = run_experiment("SGD", optim.SGD, custom_lr=0.001, momentum=0.9)
results["Adam"] = run_experiment("Adam", optim.Adam, custom_lr=0.0001)
results["AdamW"] = run_experiment("AdamW", optim.AdamW, custom_lr=0.0001)


Training with SGD

Training with Adam

Training with AdamW


In [9]:
# 실험 실행
results = {}

# 방법 1: 단순히 작은 learning rate
print("\n" + "="*70)
print("STRATEGY 1: Uniform Small Learning Rate")
print("="*70)
results["Adam_0.0001"] = run_experiment_differential_lr(
    "Adam (uniform lr=0.0001)",
    optim.Adam,
    feature_lr=0.0001,
    classifier_lr=0.0001
)

results["AdamW_0.0001"] = run_experiment_differential_lr(
    "AdamW (uniform lr=0.0001)",
    optim.AdamW,
    feature_lr=0.0001,
    classifier_lr=0.0001
)

# 방법 2: Differential learning rate (feature는 더 작게)
print("\n" + "="*70)
print("STRATEGY 2: Differential Learning Rate")
print("="*70)
results["Adam_diff"] = run_experiment_differential_lr(
    "Adam (differential)",
    optim.Adam,
    feature_lr=0.00001,   # feature는 10배 작게
    classifier_lr=0.0001  # classifier는 정상
)

results["AdamW_diff"] = run_experiment_differential_lr(
    "AdamW (differential)",
    optim.AdamW,
    feature_lr=0.00001,
    classifier_lr=0.0001
)

print("\n" + "="*60 + "\nFINAL RESULTS\n" + "="*60)
print(f"{'Optimizer':<20} {'Test Acc':<12} {'Time':<12}")
print("-" * 60)
for name, r in results.items():
    print(f"{name:<20} {r['final_acc']:>6.2f}%     {r['time']:>8.1f}s")


STRATEGY 1: Uniform Small Learning Rate


NameError: name 'run_experiment_differential_lr' is not defined

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import time

# ============================================================
# 1. 설정
# ============================================================
BATCH_SIZE = 64
EPOCHS = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ============================================================
# 2. 데이터 준비
# ============================================================
transform_train = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])
transform_test = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

print("Loading STL-10 dataset...")
train_dataset = datasets.STL10(root="./data", split="train", download=True, transform=transform_train)
test_dataset = datasets.STL10(root="./data", split="test", download=True, transform=transform_test)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")

# ============================================================
# 3. 함수 정의
# ============================================================
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()

        # Gradient clipping 추가 (폭발 방지)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        total_loss += loss.item()
        _, pred = outputs.max(1)
        total += targets.size(0)
        correct += pred.eq(targets).sum().item()
    return total_loss / len(loader), 100. * correct / total

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            total_loss += loss.item()
            _, pred = outputs.max(1)
            total += targets.size(0)
            correct += pred.eq(targets).sum().item()
    return total_loss / len(loader), 100. * correct / total

def run_experiment_differential_lr(name, opt_class, feature_lr, classifier_lr, **kwargs):
    """
    Feature extractor와 Classifier에 다른 learning rate 적용
    """
    print(f"\n{'='*50}")
    print(f"Training with {name}")
    print(f"Feature LR: {feature_lr}, Classifier LR: {classifier_lr}")
    print('='*50)

    model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
    model.classifier[6] = nn.Linear(4096, 10)
    model = model.to(DEVICE)

    criterion = nn.CrossEntropyLoss()

    # 파라미터 그룹을 나눠서 optimizer 생성
    optimizer = opt_class([
        {'params': model.features.parameters(), 'lr': feature_lr},
        {'params': model.classifier.parameters(), 'lr': classifier_lr}
    ], **kwargs)

    history = []
    start = time.time()
    for epoch in range(EPOCHS):
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer)
        te_loss, te_acc = evaluate(model, test_loader, criterion)
        history.append((tr_loss, tr_acc, te_loss, te_acc))
        print(f"Epoch {epoch+1}/{EPOCHS} | "
              f"Train Loss: {tr_loss:.4f}, Acc: {tr_acc:.2f}% | "
              f"Test Loss: {te_loss:.4f}, Acc: {te_acc:.2f}%")
    elapsed = time.time() - start
    print(f"Time: {elapsed:.2f}s, Final Test Acc: {te_acc:.2f}%")
    return {"history": history, "time": elapsed, "final_acc": te_acc}

# ============================================================
# 4. 실험 실행
# ============================================================
results = {}

# 방법 1: 단순히 작은 learning rate
print("\n" + "="*70)
print("STRATEGY 1: Uniform Small Learning Rate")
print("="*70)

results["Adam_0.0001"] = run_experiment_differential_lr(
    "Adam (uniform lr=0.0001)",
    optim.Adam,
    feature_lr=0.0001,
    classifier_lr=0.0001
)

results["AdamW_0.0001"] = run_experiment_differential_lr(
    "AdamW (uniform lr=0.0001)",
    optim.AdamW,
    feature_lr=0.0001,
    classifier_lr=0.0001
)

# 방법 2: Differential learning rate (feature는 더 작게)
print("\n" + "="*70)
print("STRATEGY 2: Differential Learning Rate")
print("="*70)

results["Adam_diff"] = run_experiment_differential_lr(
    "Adam (differential)",
    optim.Adam,
    feature_lr=0.00001,   # feature는 10배 작게
    classifier_lr=0.0001  # classifier는 정상
)

results["AdamW_diff"] = run_experiment_differential_lr(
    "AdamW (differential)",
    optim.AdamW,
    feature_lr=0.00001,
    classifier_lr=0.0001
)

# ============================================================
# 5. 최종 결과 출력
# ============================================================
print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
print(f"{'Optimizer':<20} {'Test Acc':<12} {'Time':<12}")
print("-" * 60)
for name, r in results.items():
    print(f"{name:<20} {r['final_acc']:>6.2f}%     {r['time']:>8.1f}s")

print("\n" + "="*60)
print("COMPARISON WITH ORIGINAL SGD")
print("="*60)
print(f"SGD (original):      95.74%     1104.8s")
for name, r in results.items():
    print(f"{name:<20} {r['final_acc']:>6.2f}%     {r['time']:>8.1f}s")

Using device: cuda
Loading STL-10 dataset...
Train: 5000, Test: 8000

STRATEGY 1: Uniform Small Learning Rate

Training with Adam (uniform lr=0.0001)
Feature LR: 0.0001, Classifier LR: 0.0001
Epoch 1/10 | Train Loss: 0.5032, Acc: 82.66% | Test Loss: 0.2656, Acc: 91.64%
Epoch 2/10 | Train Loss: 0.1800, Acc: 94.06% | Test Loss: 0.3060, Acc: 90.10%
Epoch 3/10 | Train Loss: 0.1292, Acc: 95.90% | Test Loss: 0.2163, Acc: 92.95%
Epoch 4/10 | Train Loss: 0.1097, Acc: 96.74% | Test Loss: 0.3044, Acc: 92.26%
Epoch 5/10 | Train Loss: 0.0692, Acc: 97.84% | Test Loss: 0.2524, Acc: 92.55%
Epoch 6/10 | Train Loss: 0.0767, Acc: 97.78% | Test Loss: 0.4565, Acc: 90.42%
Epoch 7/10 | Train Loss: 0.0623, Acc: 97.96% | Test Loss: 0.4219, Acc: 90.79%
Epoch 8/10 | Train Loss: 0.0598, Acc: 98.28% | Test Loss: 0.2733, Acc: 92.85%
Epoch 9/10 | Train Loss: 0.0657, Acc: 98.02% | Test Loss: 0.2759, Acc: 92.81%
Epoch 10/10 | Train Loss: 0.0778, Acc: 98.44% | Test Loss: 0.5285, Acc: 90.84%
Time: 1127.85s, Final Test 